# Vault Kubernetes Auth (Service Account) 설명
HashiCorp 공식 문서 - https://developer.hashicorp.com/vault/docs/auth/kubernetes  

이 노트북은 Vault의 Kubernetes 인증 방식 중 Service Account 토큰을 이용한 방식을 다룹니다.

VSO(Vault Secrets Operator) 와 K8s 인증 방식을 연동해 쉽게 Application 에서 Vault 를 연동하는 예시를 보여줍니다.  

1. **샘플 Application 생성**
2. **Vault Reviewer ServiceAccount 설정**
3. **Vault Kubernetes Auth 설정**
4. **Kubernetes 리소스 생성 (VSO 연동)**
5. **리소스 삭제 (Cleanup)**


## 환경변수 설정

In [ ]:
# direnv 환경변수 로드
direnv allow
eval $(direnv export bash)

unset AWS_ACCESS_KEY_ID
unset AWS_SECRET_ACCESS_KEY
unset AWS_SESSION_TOKEN

export SERVICE_NAME="sample-app"
export NAMESPACE="sample-app-ns"
export VAULT_NAMESPACE="vault"

# Vault 설정
export VAULT_AUTH_PATH="${SERVICE_NAME}-k8s-auth"
export VAULT_ROLE_NAME="${SERVICE_NAME}-role"
export VAULT_POLICY_NAME="${SERVICE_NAME}-policy"

# 연동할 KV 설정
export KV_MOUNT_PATH="my-kv"
export KV_PATH="app/credentials"

# K8s 설정
export K8S_HOST=$(kubectl config view --minify --output 'jsonpath={.clusters[0].cluster.server}')
export SERVICE_ACCOUNT_NAME="${SERVICE_NAME}-sa"
export AUDIENCE=${K8S_HOST}

echo "K8S_HOST: ${K8S_HOST}"
echo "NAMESPACE: ${NAMESPACE}"
echo "VAULT_AUTH_PATH: ${VAULT_AUTH_PATH}"
echo "VAULT_ROLE_NAME: ${VAULT_ROLE_NAME}"
echo "VAULT_POLICY_NAME: ${VAULT_POLICY_NAME}"
echo "SERVICE_ACCOUNT_NAME: ${SERVICE_ACCOUNT_NAME}"

## 1. 샘플 Application 생성

In [ ]:
# Namespace 생성
kubectl create namespace ${NAMESPACE}

# App 이 사용할 ServiceAccount 생성
kubectl create serviceaccount ${SERVICE_ACCOUNT_NAME} -n ${NAMESPACE}

# sample-app 생성
kubectl apply -f - <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ${SERVICE_NAME}
  namespace: ${NAMESPACE}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: ${SERVICE_NAME}
  template:
    metadata:
      labels:
        app: ${SERVICE_NAME}
    spec:
      serviceAccountName: ${SERVICE_ACCOUNT_NAME}
      containers:
      - name: main
        image: alpine
        command: ["sh", "-c", "while true; do ls -l /etc/secrets; cat /etc/secrets/*; sleep 10; done"]
        volumeMounts:
        - name: secret-volume
          mountPath: /etc/secrets
          readOnly: true
      volumes:
      - name: secret-volume
        secret:
          secretName: ${SERVICE_NAME}-k8s-secret
EOF

## 2. Vault Reviewer ServiceAccount 설정

Vault가 Kubernetes API 서버에 토큰 유효성을 검증(Token Review)할 때 사용할 권한이 필요합니다.

In [ ]:
# Reviewer SA 생성
kubectl apply -f - <<EOF
apiVersion: v1
kind: ServiceAccount
metadata:
  name: vault-reviewer
  namespace: ${VAULT_NAMESPACE}
EOF

# ClusterRoleBinding 설정 (auth-delegator 권한 부여)
kubectl apply -f - <<EOF
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: vault-reviewer-auth-delegator
subjects:
- kind: ServiceAccount
  name: vault-reviewer
  namespace: ${VAULT_NAMESPACE}
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: system:auth-delegator
EOF

### Reviewer Token 생성 (Static Token)
Vault가 사용할 수 있도록 만료되지 않는 Long-lived 토큰을 생성합니다.

In [ ]:
# Reviewer용 SA Token 생성
kubectl apply -f - <<EOF
apiVersion: v1
kind: Secret
metadata:
  name: vault-reviewer-static-token
  namespace: ${VAULT_NAMESPACE}
  annotations:
    kubernetes.io/service-account.name: "vault-reviewer"
type: kubernetes.io/service-account-token
EOF

# 생성된 토큰 환경변수에 추출
export REVIEWER_TOKEN=$(kubectl get secret vault-reviewer-static-token -n ${VAULT_NAMESPACE} -o jsonpath='{.data.token}' | base64 --decode)
export K8S_CA_CERT=$(kubectl get secret vault-reviewer-static-token -n ${VAULT_NAMESPACE} -o jsonpath='{.data.ca\.crt}' | base64 --decode)

echo "Reviewer Token extracted."

## 3. Vault Kubernetes Auth 설정
Vault 에 K8s 인증을 활성화 하고, 필요한 설정을 진행합니다.  

In [ ]:
# Role 에 할당할 Policy 생성 (KV 읽기 전용)
vault policy write sample-app-policy - <<EOF
path "${KV_MOUNT_PATH}/data/${KV_PATH}" {
  capabilities = ["read"]
}
EOF

In [ ]:
# Auth Method 활성화
vault auth enable --path=${VAULT_AUTH_PATH} kubernetes

# Config 설정 (K8s 인증 설정)
# Vault 가 받은 ServiceAccountToken 이 유효한 값인지 K8s API Server 에 요청하여 확인하기 위해 필요한 값
vault write auth/${VAULT_AUTH_PATH}/config \
    kubernetes_host="${K8S_HOST}" \
    kubernetes_ca_cert="${K8S_CA_CERT}" \
    token_reviewer_jwt="${REVIEWER_TOKEN}" \
    disable_local_ca_jwt=false

# Role 생성
# 많은 예제에서 Audience 가 빠져있는데, 최신 버전부터 Audience 는 필수값이 될 예정으로 설정 권장
vault write auth/${VAULT_AUTH_PATH}/role/${VAULT_ROLE_NAME} \
    bound_service_account_names="${SERVICE_ACCOUNT_NAME}" \
    bound_service_account_namespaces="${NAMESPACE}" \
    policies="${VAULT_POLICY_NAME}" \
    audience="${AUDIENCE}" \
    ttl="10h"

## 4. Kubernetes 리소스 생성 (VSO 연동)
위에서 만든 설정을 사용할 수 있도록 실제 K8s 에 필요한 Manifest 를 생성합니다.  
VSO 를 통해 쉽게 설정할 수 있습니다.  
https://developer.hashicorp.com/vault/docs/deploy/kubernetes/vso/api-reference 에서 자세한 스펙을 찾아볼 수 있습니다.  


In [ ]:
# VaultConnection - Endpoint 관리라고 생각하면 됨
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultConnection
metadata:
  name: vault-connection
  namespace: ${NAMESPACE}
spec:
  address: ${VAULT_ADDR}
EOF

# VaultAuth - Vault 인증 관리
# method 에 따라 인증 방식 변경 가능
# 여기서는 kubernetes 이며, spec/kubernetes 안에서 세부적인 설정 필요함
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultAuth
metadata:
  name: ${SERVICE_NAME}-auth
  namespace: ${NAMESPACE}
spec:
  method: kubernetes
  mount: ${VAULT_AUTH_PATH}
  kubernetes:
    role: ${VAULT_ROLE_NAME}
    serviceAccount: ${SERVICE_ACCOUNT_NAME}
    audiences: 
      - ${AUDIENCE}
  vaultConnectionRef: vault-connection
EOF

# VaultStaticSecret - 실제로 VSO 가 연동할 Secret 정보 관리
# 여기서는 kv 를 연동했음.
# 리스와 같은 리소스는 VaultDynamicSecret 으로 연동 가능
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  name: ${SERVICE_NAME}-secret
  namespace: ${NAMESPACE}
spec:
  vaultAuthRef: ${SERVICE_NAME}-auth
  
  mount: ${KV_MOUNT_PATH}
  type: kv-v2
  path: ${KV_PATH}
  destination:
    name: ${SERVICE_NAME}-k8s-secret
    create: true
  refreshAfter: 30s
  
  rolloutRestartTargets:
  - kind: Deployment
    name: ${SERVICE_NAME}
EOF

## 5. 리소스 삭제 (Cleanup)

테스트를 위해 생성한 모든 리소스를 삭제합니다.

In [ ]:
# Kubernetes 리소스 삭제
kubectl delete deployment ${SERVICE_NAME} -n ${NAMESPACE}
kubectl delete vaultstaticsecret ${SERVICE_NAME}-secret -n ${NAMESPACE}
kubectl delete vaultauth ${SERVICE_NAME}-auth -n ${NAMESPACE}
kubectl delete vaultconnection vault-connection -n ${NAMESPACE}
kubectl delete serviceaccount ${SERVICE_ACCOUNT_NAME} -n ${NAMESPACE}
kubectl delete namespace ${NAMESPACE}

# Reviewer 리소스 삭제
kubectl delete secret vault-reviewer-static-token -n ${VAULT_NAMESPACE}
kubectl delete clusterrolebinding vault-reviewer-auth-delegator
kubectl delete serviceaccount vault-reviewer -n ${VAULT_NAMESPACE}

# Vault Auth 설정 삭제
vault auth disable ${VAULT_AUTH_PATH}